## Load cleaned data

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report

df = pd.read_csv('outputs/cleaned_tickets.csv')
df = df.dropna(subset=['clean_text'])
df.shape

(8469, 4)

## Category model - train test split

In [2]:
X = df['clean_text']
y_cat = df['category']

X_train, X_test, y_cat_train, y_cat_test = train_test_split(X, y_cat, test_size=0.2, random_state=42)
X_train.shape, X_test.shape

((6775,), (1694,))

## TF-IDF vectorization

In [3]:
tfidf = TfidfVectorizer(max_features=3000)
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)
X_train_vec.shape

(6775, 3000)

## Train category classifier

In [4]:
cat_model = LogisticRegression(max_iter=1000)
cat_model.fit(X_train_vec, y_cat_train)
cat_preds = cat_model.predict(X_test_vec)

## Evaluate category model

In [5]:
print('accuracy:', accuracy_score(y_cat_test, cat_preds))
print(classification_report(y_cat_test, cat_preds))

accuracy: 0.19539551357733176
                      precision    recall  f1-score   support

     Billing inquiry       0.18      0.13      0.15       357
Cancellation request       0.19      0.19      0.19       327
     Product inquiry       0.18      0.19      0.19       316
      Refund request       0.19      0.21      0.20       345
     Technical issue       0.22      0.25      0.24       349

            accuracy                           0.20      1694
           macro avg       0.19      0.20      0.19      1694
        weighted avg       0.19      0.20      0.19      1694



## Priority model - train test split

In [6]:
y_pri = df['priority']

X_train2, X_test2, y_pri_train, y_pri_test = train_test_split(X, y_pri, test_size=0.2, random_state=42)
X_train2.shape, X_test2.shape

((6775,), (1694,))

## TF-IDF for priority (same vectorizer settings, fit again on this split)

In [7]:
tfidf_pri = TfidfVectorizer(max_features=3000)
X_train2_vec = tfidf_pri.fit_transform(X_train2)
X_test2_vec = tfidf_pri.transform(X_test2)
X_train2_vec.shape

(6775, 3000)

## Train priority classifier

In [8]:
pri_model = LogisticRegression(max_iter=1000)
pri_model.fit(X_train2_vec, y_pri_train)
pri_preds = pri_model.predict(X_test2_vec)

## Evaluate priority model

In [9]:
print('accuracy:', accuracy_score(y_pri_test, pri_preds))
print(classification_report(y_pri_test, pri_preds))

accuracy: 0.2650531286894923
              precision    recall  f1-score   support

    Critical       0.29      0.30      0.30       411
        High       0.24      0.26      0.25       409
         Low       0.25      0.23      0.24       415
      Medium       0.28      0.27      0.28       459

    accuracy                           0.27      1694
   macro avg       0.26      0.27      0.26      1694
weighted avg       0.27      0.27      0.26      1694



## Save predictions for visualization notebook

In [10]:
results_df = pd.DataFrame({
    'clean_text': X_test.values,
    'actual_category': y_cat_test.values,
    'predicted_category': cat_preds
})
results_df.to_csv('outputs/category_predictions.csv', index=False)
results_df.head()

,clean_text,actual_category,predicted_category
0,product setup im issue please assist im using ...,Refund request,Technical issue
1,battery life im trouble connecting home wifi n...,Product inquiry,Product inquiry
2,refund request im issue please assist please g...,Billing inquiry,Cancellation request
3,peripheral compatibility im issue please assis...,Billing inquiry,Cancellation request
4,peripheral compatibility im issue please assis...,Refund request,Billing inquiry


In [11]:
priority_results_df = pd.DataFrame({
    'clean_text': X_test2.values,
    'actual_priority': y_pri_test.values,
    'predicted_priority': pri_preds
})
priority_results_df.to_csv('outputs/priority_predictions.csv', index=False)
priority_results_df.head()

,clean_text,actual_priority,predicted_priority
0,product setup im issue please assist im using ...,High,Critical
1,battery life im trouble connecting home wifi n...,Low,Low
2,refund request im issue please assist please g...,Low,Low
3,peripheral compatibility im issue please assis...,High,Low
4,peripheral compatibility im issue please assis...,Medium,Critical
